In [1]:
import time
import pandas as pd
import numpy as np
import sqlite3

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html
import dash_bootstrap_components as dbc


source_data_path = r"P:\Application\Risk Mgmt\MRM\ARMS\bonds_report\bond_portfolio_report.xlsx"
basel_buckets =['1D','1M','3M','6M','9M','1Y','1.5Y','2Y','3Y','4Y','5Y','6Y','7Y','8Y','9Y','10Y','15Y','20Y','30Y']

## Help

[Dash Documentation](https://dash.plotly.com/)

[Dash Bootstrap Components](https://www.dash-bootstrap-components.com/)

[Bootstrap Themes](https://bootswatch.com/)

[Plotly Color Reference](https://masamasace.github.io/plotly_color/)

[HTML Tutorials](https://www.w3schools.com/html/)

[Plotly Theming and templates](https://plotly.com/python/templates/)

[Code from the Book 'Interactive-Dashboards-and-Data-Apps-with-Plotly-and-Dash'](https://github.com/PacktPublishing/Interactive-Dashboards-and-Data-Apps-with-Plotly-and-Dash/)

[A4 size in pixels](https://www.a4-size.com/a4-size-in-pixels/)


## Import data as DataFrames

In [2]:
start_time_total = time.time()
start_time = time.time()
#-------------------------------------------------------------------------------------------

# Dictionary mapping sheet names to desired DataFrame variable names
sheet_to_df_name = {
    'metrics_sum': 'df_metrics_sum',
    'metrics': 'df_metrics',
    'by_rating': 'df_rating',
    'by_sector': 'df_sector',
    'cash_flow': 'df_cf',
    'pnl_port': 'df_pnl_port',
    'new_deals': 'df_new_deals',
    'source_data': 'source_data',
    'cf_data': 'cf_data',
    'rating_matrix': 'rating_matrix'
}

# Dictionary to hold the actual DataFrames
dataframes = {}


# Load each sheet into the corresponding DataFrame
for sheet_name, df_name in sheet_to_df_name.items():
    try:
        dataframes[df_name] = pd.read_excel(source_data_path, sheet_name=sheet_name)
        print(f"size of {df_name} is {dataframes[df_name].shape}")
    except Exception as e:
            print(f"failed to read list {sheet_name}: {e}")

# assign each DataFrame to a variable in the global namespace
globals().update(dataframes)

# get dates that are in report data
dates_array = source_data['RepDate'].unique()
dates_array = sorted(dates_array)
start_date = dates_array[0]
end_date = dates_array[-1]
selected_dates = [dates_array[i] for i in [0,-21,-6,-1]]
prev_date1 = selected_dates[2]

#create dictionary metric for better readability
dic_metrics = {
    'book_value':'Book Value',
    'fair_value': 'Fair Value',
    'mod_duration': 'Modified Duration',
    'expect_loss': 'Expected Loss (lifetime)',
    'unexpect_loss': 'Unexpected Loss (lifetime)',
    'pnl_azn': 'Daily PnL (AZN)',
    'return_cum_pp': 'Time Weighted Return (PTD)',
    'parallel_up':'Parallel Up', 
    'parallel_down':'Parallel Down', 
    'short_rates_up':'Short Rates Up',
    'short_rates_down':'Short Rates Down', 
    'steepener':'Steepener', 
    'flattener':'Flattener',
    'parallel_up_bp':'Parallel Up', 
    'parallel_down_bp':'Parallel Down', 
    'short_rates_up_bp':'Short Rates Up',
    'short_rates_down_bp':'Short Rates Down', 
    'steepener_bp':'Steepener', 
    'flattener_bp':'Flattener',
    'stress_covid': 'COVID 2020', 
    'stress_inflation': 'Inflation 2021',
}

#-------------------------------------------------------------------------------------------
end_time = time.time()
print(f"Execution time: {end_time - start_time:.4f} seconds")

size of df_metrics_sum is (205, 68)
size of df_metrics is (301, 69)
size of df_rating is (33, 68)
size of df_sector is (20, 5)
size of df_cf is (260, 11)
size of df_pnl_port is (42, 67)
size of df_new_deals is (3, 11)
size of source_data is (14182, 74)
size of cf_data is (1469, 5)
size of rating_matrix is (22, 18)
Execution time: 12.1651 seconds


## [Example - Amazon](https://github.com/mayaradaher/challenge-Amazon/blob/main/app.py/)
## [Dash Tutorial](https://open-resources.github.io/dash_curriculum/preface/about.html/)

## Chart functions

In [3]:
start_time = time.time()
#-------------------------------------------------------------------------------------------

from function_metrics_calculation import cf_aggregation
from functions_yield_curves import get_par_yield_curve
from functions_misc import get_dm_credit_spread, get_em_credit_spread
from functions_yield_curves import basel_rate_shocks

#=============================plot_book_value_and_mod_duration===============================
def plot_book_value_and_mod_duration(currency):
    """
    stacked bar chart for portfolios on the left Y-axis and 
    line charts for Modified duration on the right Y-axis
    """
    metric_list = ['book_value', 'mod_duration']
    df = df_metrics_sum.loc[df_metrics_sum['metric'].isin(metric_list)].copy()
    # make order for stacking data
    port_order = ['AFS PORTFOLIO', 'HTM PORTFOLIO', 'MORTGAGE BONDS']
    
    df['portfolio_type'] = pd.Categorical(
        df['portfolio_type'],
        categories=port_order,
        ordered=True)

    # --- X tick reduction:
    #step = max(1, int(np.ceil(len(dates_array))/5))
    step = 10
    tickvalues = dates_array[::step]
    
    # charting    
    df_crnc = df.loc[df['currency'] == currency]
    df_book_value =  df_crnc.loc[df_crnc['metric'] == metric_list[0]]
    df_book_value = df_book_value.sort_values('portfolio_type')
    df_mod_daration = df_crnc.loc[df_crnc['metric'] == metric_list[1]]
    df_mod_daration = df_mod_daration.sort_values('portfolio_type')

    fig = go.Figure()
    
    #Book Value - Stacked Bars
    for i, (_, row) in enumerate(df_book_value.iterrows()):
        fig.add_trace(go.Bar(
            x=dates_array,
            y=row[dates_array],
            name=f"{row['portfolio_type']}",
            yaxis='y1',
            # add legend group
            legendgroup=metric_list[0],                       # group id
            legendgrouptitle_text=dic_metrics[metric_list[0]] if i == 0 else None,  # show title once
            legendrank=10                                   # order this group before the other
            )
        )

    # Mod Duration - Lines
    for i, (_, row) in enumerate(df_mod_daration.iterrows()):
        fig.add_trace(go.Scatter(
            x=dates_array,
            y=row[dates_array],
            name=f"{row['portfolio_type']}",
            yaxis='y2',
            mode='lines',
            # add legend group
            legendgroup=metric_list[1],
            legendgrouptitle_text=dic_metrics[metric_list[1]] if i == 0 else None,
            legendrank=20
            )
        )
    
    # Layout
    fig.update_layout(
        title=dict(
            text=f"Book Value and Modified Duration for bonds denominated in {currency}",
            font=dict(size=30)),
        xaxis=dict(
            #title='Date', 
            type='category',                  
            tickmode='array',
            tickvals=tickvalues),
        yaxis=dict(title='Book Value, AZN', side='left'),
        yaxis2=dict(title='Mod Duration', overlaying='y', side='right'),
        barmode='stack',
        legend=dict(orientation='h', 
                    x=0.5, # center horizontally
                    xanchor='center',
                    y=-0.25,
                    yanchor='top', 
                    traceorder='grouped',      # keep traces grouped in legend
                    tracegroupgap=12,          # space between groups
                    ),

        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
        # to avoid clipping the legend
        #margin=dict(b=60),  
        # hover across traces
        #hovermode='x unified'
    )
    return fig

#==============plot_cash_flow======================================================
def plot_cash_flow(portfolio, currency, fx_translation = True):

    # get data
    df = cf_aggregation (end_date, source_data, cf_data, 
                   aggregate_port = portfolio, # portfolio type or 'all'
                   aggregate_currency = currency, # currency code or AZN 
                   aggregate_time = 'quarter' # alternative 'month_end'
    )
    df_prev = cf_aggregation (end_date, source_data, cf_data, 
               aggregate_port = portfolio, # portfolio type or 'all'
               aggregate_currency = currency, # currency code or AZN 
               aggregate_time = 'quarter' # alternative 'month_end'
    ) 
    col_azn_translation = ['time_period', 'coupon_azn', 'principal_azn', 'principal_remain_azn']
    col_as_is = ['time_period', 'coupon', 'principal', 'principal_remain']

    if fx_translation:
        columns_to_take =  col_azn_translation
        yaxis_name_crnc='AZN'
    else:
        columns_to_take = col_as_is
        yaxis_name_crnc='original currency'
    
    df = df[columns_to_take]
    #rename columns
    df.columns = col_as_is
    # --- X tick reduction:
    dates_array = df['time_period'].tolist()
    #step = max(1, int(np.ceil(len(dates_array))/2))
    step = 2
    tickvalues = dates_array[::step]

    # create plot
    fig = go.Figure()

    # add principal as bar
    fig.add_trace(
        go.Bar(
            x=df['time_period'],
            y=df['principal'],
            name='principal',
            yaxis='y1')      
    )
    # add coupon as bar
    fig.add_trace(
        go.Bar(
            x=df['time_period'],
            y=df['coupon'],
            name='coupon',
            yaxis='y1')      
    )
    # add previous outstanding principal as line (secondary axis)
    fig.add_trace(
        go.Scatter(
        x=df_prev['time_period'],
        y=df_prev['principal_remain'],
        name="outstanding principal previous",
        yaxis='y2',
        mode='lines',
        line=dict(dash='dash')
        )
    )
    # add current outstanding principal as line (secondary axis)
    fig.add_trace(
        go.Scatter(
        x=df['time_period'],
        y=df['principal_remain'],
        name="outstanding principal current",
        yaxis='y2',
        mode='lines'
        )
    )
    
    # specify layout
    fig.update_layout(
        title=dict(
            text=f'{currency} {portfolio} Cash Flow',
            font=dict(size=25)
        ),
        xaxis=dict(
            title='time period',
            type='category',
            tickmode='array',
            tickvals=tickvalues
        ),
        yaxis=dict(title=f'Cash Flow, {yaxis_name_crnc}', side='left'),
        yaxis2=dict(title=f'Outstandig Principal, {yaxis_name_crnc}', overlaying='y', side='right'),
        barmode='stack',
        legend=dict(
            x=1,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(0,0,0,0)'
        ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )
    return fig

#==============plot_ratings_and_risk======================================================
def plot_ratings_and_risk(currency, portfolio='all', presentation ='absolute'):
    """
    present = 'absolute' or 'relative'
    """
    # get rating data
    df_cr = df_rating.loc[df_rating['currency']==currency].copy()  
    if portfolio !='all':
        df_cr = df_cr.loc[df_cr['portfolio_type']==portfolio].copy() 
    # group by rating category
    df_cr = df_cr.groupby(['rating_category'], observed=True)[dates_array].sum().reset_index()
    df_cr = df_cr.sort_values(by='rating_category', ascending=False)

    # get metrics data
    metrics = ['expect_loss','unexpect_loss']
    df_ratios = df_metrics_sum.loc[
        (df_metrics_sum['currency']==currency)&
        (df_metrics_sum['metric']).isin(metrics)].copy()  
    if portfolio !='all':
        df_ratios = df_ratios.loc[df_ratios['portfolio_type']==portfolio].copy()
    else:
        df_ratios['portfolio_type']='all'
    # group by portfolio
    df_ratios = df_ratios.groupby(['metric', 'portfolio_type'], observed=True)[dates_array].sum().reset_index()

    # preparatory transformation 
    # ticks
    step = 10
    tickvalues = dates_array[::step]
    
    # adjust to portfolio selection
    if portfolio=='all':
        title_chart = f'Credit quality and risk of {currency} bonds'
    else:
        title_chart = f'Credit quality and risk of {currency} {portfolio}'
    
    # adjust to presentation selection
    denominator = df_cr[dates_array].sum()/100
    
    if presentation == 'absolute':
        title_yaxis = 'Rating Categories, AZN'
        title_yaxis2 = 'Loss Metric, AZN'
        denominator = 1
        title_chart = f'{title_chart}: Absolute Values'
    else:
        title_yaxis = 'Rating Categories, %'
        title_yaxis2 = 'Loss Metric, %'
        title_chart = f'{title_chart}: Relative Values'
    # create plot
    fig = go.Figure()

    # Define consistent color scheme
    color_map = {
        'IG1 Aaa': 'deepskyblue',
        'IG2 Aa': 'limegreen',
        'IG3 A': 'greenyellow',
        'IG4 Baa': 'lightgreen',
        'N-IG1 Ba': 'khaki',
        'N-IG2 B': 'darkorange',
        'N-IG3 Caa': 'deeppink',
        'N-IG4 C': 'magenta',
        'Non-Rated': 'orchid',
    }
    
    # add rating categories as bars
    for index, row in df_cr.iterrows():
        trace_name = row['rating_category']
        fig.add_trace(
            go.Bar(
                name=trace_name,
                x=dates_array,
                y=row[dates_array]/denominator,
                yaxis='y1',
                marker=dict(color=color_map.get(trace_name))
            )
        )

    # add expected and unexpected loss
    color_schema=['royalblue', 'red']
    for i, metric in enumerate(metrics):
        fig.add_trace(
            go.Scatter(
                name=dic_metrics[metric],
                x=dates_array,
                y=df_ratios.loc[df_ratios['metric']==metric, dates_array].values.flatten()/denominator,             
                yaxis='y2',
                line=dict(color=color_schema[i], width=3, dash='solid')
            )
        )
    
    # specify layout
    fig.update_layout(
        title=dict(
            text=title_chart,
            font=dict(size=25)
        ),
        xaxis=dict(
            #title='Date',
            type='category',
            tickmode='array',
            tickvals=tickvalues
        ),
        yaxis=dict(
            title=title_yaxis,
            side='left'
        ),
        yaxis2=dict(
            title=title_yaxis2,
            side='right',
            overlaying='y'
        ),
        
        barmode= 'stack',
        legend=dict(
            orientation='h', 
            x=0.5, # center horizontally
            xanchor='center',
            y=-0.25,
            yanchor='top', 
            ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )

    return fig

#==============plot_int_rate_risk======================================================
def plot_int_rate_risk (portfolio='all', currency='FCY'):
    """
    currency = 'AZN' or 'FCY'
    portfolio = any name in position report or 'all'
    """
    # filter currency
    df = df_metrics_sum.loc[df_metrics_sum['currency']==currency].copy()
    #filter metrics    
    metrics = df['metric'].unique()
    metrics = [metric for metric in metrics if 'dv01_' in metric]
    metrics = ['cr01'] + metrics
    df = df.loc[df['metric'].isin(metrics)]

    # adjust to portfolio selection
    if portfolio=='all':
        title_chart = f'Market Risk of {currency} bonds'
        df['portfolio_type']='all'
    else:
        title_chart = f'Market Risk of {currency} {portfolio}'
        df = df.loc[df['portfolio_type']==portfolio]

    # group data
    df = df.groupby(['metric','portfolio_type'], observed=True)[dates_array].sum().reset_index()
    # filter out empty zero dv buckets and sort by value
    df = df.loc[
        (df['metric']=='cr01')|
        (df[dates_array].sum(axis=1)!=0)
    ]
    df = df.sort_values(by=dates_array[-1], ascending=True)
    
    # ticks
    step = 10
    tickvalues = dates_array[::step]
    
    # create plot
    fig = go.Figure()

    for _, row in df.iterrows():
        if row['metric'] =='cr01':
            fig.add_trace(
                go.Scatter(
                    name=row['metric'],
                    x=dates_array,
                    y=row[dates_array],
                    mode='lines',
                    line=dict(color='red', width=3, dash='solid')
                )
            )
        else:
            fig.add_trace(
                go.Bar(
                    name=row['metric'],
                    x=dates_array,
                    y=row[dates_array]
                )
            )
    # set layout
    fig.update_layout(
        title=dict(
            text=title_chart,
            font=dict(size=25)
        ),
        xaxis=dict(
            #title='date',
            type='category',
            tickmode='array',
            tickvals=tickvalues
        ),
        yaxis=dict(
            title='Change in fair value, AZN',
            side='left'
        ),
        barmode='stack',
        legend=dict(
            orientation='h', 
            x=0.5, # center horizontally
            xanchor='center',
            y=-0.25,
            yanchor='top', 
            ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',    
    )
    
    return fig

#==============plot_em_credit_spread======================================================
def plot_em_credit_spread(selected_dates, metric,bucket):
    """
    Chart option adjusted credit spread for selected ratings
    """
    db_path = 'arms_database.db'
    rating_list = ['A1', 'A2', 'A3', 'Baa1', 'Baa2', 'Baa3', 
                   'Ba1', 'Ba2', 'Ba3',	'B1', 'B2']

    # create plot
    fig=go.Figure()

    for i, date in enumerate(selected_dates):
        df = get_em_credit_spread(date, metric, bucket, db_path)
        df[rating_list]=df[rating_list].round(0)
        y_values = df[rating_list].values.flatten()
        show_labels = i in [0, 3] # Add data labels only for the first and fourth traces
        fig.add_trace(
            go.Bar(
                x=rating_list,
                y= y_values,
                name = f"{date}",
                yaxis='y1', 
                text=y_values if show_labels else None,
                textposition='outside' if show_labels else None 
            )
        )
    # specify layout 
    fig.update_layout(
        title = dict(
            text = f"Emerging Markets: Option Adjusted Credit Spread ({metric}) <br>for USD bonds, Time {bucket} years",
            font=dict(size=25)
        ),
        yaxis=dict(
            title="credit spread, bp",
            side="right"
        ),
        barmode='group',
        legend=dict(
            orientation='v', 
            x=0,
            y=1,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(0,0,0,0)'  # transparent background
        ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )    
    return fig

#==============plot_dm_credit_spread======================================================
def plot_dm_credit_spread(selected_dates, metric,bucket):
    """
    Chart option adjusted credit spread for selected ratings
    """
    db_path = 'arms_database.db'
    rating_list = ['Aa2', 'Aa3', 'A1', 'A2', 'A3', 'Baa1', 'Baa2', 'Baa3', 
               'Ba1', 'Ba2', 'Ba3']

    # create plot
    fig=go.Figure()

    for i, date in enumerate(selected_dates):
        df = get_dm_credit_spread(date, metric, bucket, db_path)
        df[rating_list]=df[rating_list].round(0)
        y_values = df[rating_list].values.flatten()
        show_labels = i in [0, 3] # Add data labels only for the first and fourth traces
        fig.add_trace(
            go.Bar(
                x=rating_list,
                y= y_values,
                name = f"{date}",
                yaxis='y1', 
                # add data labels
                text=y_values if show_labels else None,
                textposition='outside' if show_labels else None 
            )
        )
    # specify layout 
    fig.update_layout(
        title = dict(
            text = f"Developed Markets: Option Adjusted Credit Spread ({metric}) <br>for USD bonds, Time {bucket} years",
            font=dict(size=25)
        ),
        yaxis=dict(
            title="credit spread, bp",
            side="left"
        ),
        barmode='group',
        legend=dict(
            orientation='v', 
            x=0,
            y=1,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(0,0,0,0)'  # transparent background
        ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )    
    return fig

    #==============plot_par_yield_curve======================================================
def plot_par_yield_curve(selected_dates, currency, range_days):
    """
    Plot par yield curve for selected currencies and time buckets
    """
    db_path = 'arms_database.db'

    # create plot
    fig=go.Figure()

    for i, date in enumerate(selected_dates):
        df = get_par_yield_curve(date, currency, db_path)
        df= df.loc[
            (df['tenor_days']>=range_days[0])&
            (df['tenor_days']<=range_days[1])
        ]
        y_values = df['rate_percent'].astype(float).round(2)
        show_labels = (i+1==len(selected_dates))
        fig.add_trace(
            go.Scatter(
                x=df['tenor'],
                y= y_values,
                name = f"{date}",
                yaxis='y1',
                mode='lines+text' if show_labels else 'lines',
                line=dict(dash='solid' if show_labels else 'dash'),
                text=y_values if show_labels else None,
                textposition='top center' if show_labels else None 
            )
        )
    # specify layout 
    fig.update_layout(
        title = dict(
            text = f"{currency} Par Yield Curve",
            font=dict(size=25)
        ),
        yaxis=dict(
            title="par yield, pp",
            side="left"
        ),
        legend=dict(
            orientation='h', 
            x=0.5, # center horizontally
            xanchor='center',
            y=-0.25,
            yanchor='top', 
            ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )    
    return fig

#========plot_pnl=================plot_pnl==========================================
def plot_pnl(portfolio, currency):
    
    #get data
    metrics = ['pnl_azn', 'return_cum_pp']
    df= df_pnl_port.loc[
        (df_pnl_port['portfolio_type']==portfolio)&
        (df_pnl_port['currency']==currency)&
        (df_pnl_port['metric'].isin(metrics))
    ].copy()
    x_values=dates_array[1:] #PnL data miss the first date in the period
    df[x_values]=df[x_values].astype(float).round(2)
    #create plot
    fig=go.Figure()

    # ticks
    step = 10
    tickvalues = x_values[::step]
    # y values
    y1_values = df.loc[df['metric']==metrics[0], x_values].values.flatten()
    y2_values = df.loc[df['metric']==metrics[1], x_values].values.flatten()

    # add daily PnL 
    fig.add_trace(go.Bar(
        name=dic_metrics[metrics[0]],
        x=x_values,
        y=y1_values,
        yaxis='y1',
        )
    )
    # add time weighted return 
    fig.add_trace(go.Scatter(
        name=dic_metrics[metrics[1]],
        x=x_values,
        y=y2_values,
        yaxis='y2',
        mode='lines'
        )
    )

    # specify layout
    fig.update_layout(
        title=dict(
            text = f"PnL: {currency} {portfolio}",
            font=dict(size=25)
        ),
        xaxis = dict(
            type='category',
            tickmode='array',
            tickvals=tickvalues
        ),
        yaxis=dict(
            title="Daily PnL, AZN",
            side='left'
        ),
        yaxis2=dict(
            title="Return, %",
            side='right',
            overlaying='y'
        ),
        legend=dict(
            orientation='h', 
            x=0.5, # center horizontally
            xanchor='center',
            y=-0.25,
            yanchor='top',
            bgcolor='rgba(0,0,0,0)'
            ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )     
    return fig

#==========plot_reval_diff======================================================
def plot_reval_diff(portfolio, currency):
    """
    portfolio as is in report or 'all'
    currecny as is in report or 'FCY' or 'all'
    """
    #get data
    metrics = ['book_value', 'fair_value']

    df= df_metrics.loc[
        df_metrics['metric'].isin(metrics)
    ].copy()
    
    # adjust to currency aggregation
    if currency=='FCY':
        df.loc[df['currency']!='AZN', 'currency']='FCY'
    elif currency=='all':
        df['currency']='all'
    elif currency not in df['currency'].unique():
        print(f"unknown currency: {currency}")
        return go.Figure()
    
    # adjust to portfolio aggregation
    if portfolio =='all':
        df['portfolio_type']='all'
    elif portfolio not in df['portfolio_type'].unique():
        print(f"unknown portfolio: {portfolio}")
        return go.Figure()

    # filter required data
    df = df.loc[
        (df['currency']==currency)&
        (df['portfolio_type']==portfolio)
    ]

    # group data
    index_columns = ['portfolio_type', 'currency', 'metric']
    df= df.groupby(index_columns, observed=True)[dates_array].sum().reset_index()

    # create plot
    fig=go.Figure()
    
    # ticks
    step = 10
    tickvalues = dates_array[::step]

    # y values
    value_y1=df.loc[df['metric']==metrics[0], dates_array].values.flatten()
    value_y2=df.loc[df['metric']==metrics[1], dates_array].values.flatten()
    value_y3 = value_y2 - value_y1

    line_values = [value_y1, value_y2]

    # add metrics
    for i, metric in enumerate(metrics):
        fig.add_trace(
            go.Scatter(
                name=dic_metrics[metric],
                x=dates_array,
                y=line_values[i],
                yaxis='y1',
                mode='lines'        
            )
        )
    # add difference
    fig.add_trace(
        go.Bar(
            name = "Difference",
            x=dates_array,
            y=value_y3,
            yaxis='y2'
        )
    )
    if portfolio =='all' and currency =='all':
        title_name = 'All Bonds'
    elif portfolio =='all' and currency !='all':
        title_name = f"{currecny} Bonds"
    elif portfolio !='all' and currency =='all':
        title_name = f" All Bonds in {portfolio}"
    else:
        title_name = f"{currency} {portfolio}"
    
            # specify layout
    fig.update_layout(
        title=dict(
            text = title_name,
            font=dict(size=25)
        ),
        xaxis = dict(
            type='category',
            tickmode='array',
            tickvals=tickvalues
        ),
        yaxis=dict(
            title="Value, AZN",
            side='left',
            overlaying='y2'
        ),
        yaxis2=dict(
            title="Difference, AZN",
            side='right',
        ),
        legend=dict(
            orientation='h', 
            x=0.5, # center horizontally
            xanchor='center',
            y=-0.25,
            yanchor='top',
            bgcolor='rgba(0,0,0,0)'
        ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',        
    )    
    return fig

#==========plot_sector_exposure===========================================================
def plot_sector_exposure(currency):
    # prepare data
    df = df_sector.loc[df_sector['currency']==currency]
    df=df.sort_values(by='end_period')
    value_columns = ['start_period', 'change', 'end_period']
    df[value_columns]=df[value_columns].round(0)

    # create plot
    fig=make_subplots(
        rows=1,cols=3, 
        subplot_titles=(f"as of {start_date}", "change", f"as of {end_date}"),
        horizontal_spacing=0.01 # space between subplots
    )
    # start period
    fig.add_trace(
        go.Bar(
            x=df['start_period'],
            y=df['sector'],
            orientation='h',           
        ), row=1, col=1
    )
    # difference
    fig.add_trace(
        go.Bar(
            x=df['change'],
            y=df['sector'],
            orientation='h',
        ),
    row=1, col=2)
    # end period
    fig.add_trace(
        go.Bar(
            x=df['end_period'],
            y=df['sector'],
            orientation='h',
        ), 
    row=1, col=3)

    #set common y-axis layouts
    fig.update_yaxes(
        type='category',
        categoryorder='array',
        categoryarray=df['sector'],
        tickmode='array',
        tickvals=df['sector'],
        showticklabels=False  # start hidden for all
    )
    # Override common layouts
    fig.update_yaxes(showticklabels=True, side='left',  row=1, col=1)  # left subplot
    fig.update_yaxes(showticklabels=True, side='right', row=1, col=3)  # right subplot

    
    fig.update_layout(
        title=dict(
            text=f"{currency} Bond Portfolio breakdown by Sector",
            font=dict(size=20)
        ),
        showlegend=False,
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',     
    )    
    return fig

#=======plot_basel_shock_scenario===========================================
def plot_basel_shock_scenario(currency):

    #====this part is taken from metric calculation notebook================
    hard_currencies = ['USD', 'EUR', 'CHF', 'GBP', 'JPY', 'CAD']
    scenarios = pd.DataFrame({
        'currency_type' : ['hard', 'other'],
        'parallel' : [200, 400],
        'short' : [300, 500],
        'long' : [ 150, 300]
    })
    #=======================================================================
    # calculate scenarios
    basel_buck=basel_buckets[:-3] #cut off two longest buckets
    
    df = basel_rate_shocks(currency, end_date, basel_buck, scenarios, hard_currencies)
    scenarios = ['parallel_up_bp', 'parallel_down_bp', 'short_rates_up_bp',
                 'short_rates_down_bp', 'steepener_bp', 'flattener_bp']

    # Define consistent color scheme
    color_map = {
        'parallel_up_bp': 'blue',
        'parallel_down_bp': 'red',
        'short_rates_up_bp': 'green',
        'short_rates_down_bp': 'orange',
        'steepener_bp': 'purple',
        'flattener_bp': 'brown'
    }
    
    # initiate chart
    fig=go.Figure()

    # add values
    for scenario in scenarios:
        fig.add_trace(
            go.Scatter(
                name=dic_metrics.get(scenario,scenario),
                x=df['buckets'],
                y=df[scenario],
                mode='lines',
                line=dict(color=color_map.get(scenario)),
            )
        )
    fig.update_layout(
        title=dict(
            text=f"Basel Rate Shocks for {currency}",
            font=dict(size=20)
        ),
        yaxis=dict(
            title="rate shift by, bp",          
            side='left'
        ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )
    return fig

#=======plot_basel_shock_results===========================================
def plot_basel_shock_results():

    # Define scenarios, currencies, and portfolio
    scenarios = ['parallel_up', 'parallel_down', 'short_rates_up',
                 'short_rates_down', 'steepener', 'flattener']
    currencies=['FCY', 'AZN']
    portfolios=['MORTGAGE BONDS', 'AFS PORTFOLIO', 'HTM PORTFOLIO']

    # Filter relevant data
    df = df_metrics_sum.loc[df_metrics_sum['metric'].isin(scenarios)]
    
    # create subplots titles
    subplot_titles = ['All Bonds'] + [
        f"{c} {p}"
        for c in currencies
        for p in portfolios
        if not (c == 'FCY' and p == 'MORTGAGE BONDS')
    ]
    subplot_titles=tuple(subplot_titles)

    # ticks
    step = 20
    tickvalues = dates_array[::step]
    
    # Create subplots
    fig=make_subplots(
        rows=2,cols=3,
        subplot_titles=(subplot_titles),
        horizontal_spacing=0.03, # space between subplots
        shared_xaxes=True,
        shared_yaxes=False   
    )
    
    # Define consistent color scheme
    color_map = {
        'parallel_up': 'blue',
        'parallel_down': 'red',
        'short_rates_up': 'green',
        'short_rates_down': 'orange',
        'steepener': 'purple',
        'flattener': 'brown'
    }

    # Plot aggregated data for all bonds in first subplot
    for scenario in scenarios:
        y_values = df.loc[df['metric']==scenario, dates_array].sum().values
        fig.add_trace(
            go.Scatter(
                name = dic_metrics.get(scenario, scenario),
                x=dates_array,
                y=y_values,
                mode='lines',
                line=dict(color=color_map.get(scenario)),
                showlegend=True
            ), row=1, col=1
        )
    
    # Plot data for each currency and portfolio combination
    subplot_index=0
    for currency in currencies:
        for portfolio in portfolios:
            # get position
            subplot_index += 1
            row = 1 if subplot_index <= 3 else 2
            col = subplot_index if row==1 else subplot_index-3
            # skip FCY Mortgage Bonds as they do not exist
            if (portfolio=='MORTGAGE BONDS' and currency=='FCY'):
                continue
            
            #plot data
            for scenario in scenarios:
                y_values =df.loc[
                    (df['metric']==scenario)&
                    (df['portfolio_type']==portfolio)&
                    (df['currency']==currency),
                dates_array].values.flatten()
                fig.add_trace(
                    go.Scatter(
                        name = dic_metrics.get(scenario, scenario),
                        x=dates_array,
                        y=y_values,
                        mode='lines',
                        line=dict(color=color_map.get(scenario)),
                        showlegend=False
                    ), row=row, col=col
                    )
    
    fig.update_layout(
        title=dict(
            text=f"Change in fair value under Basel's Rate Shock Scenarios",
            font=dict(size=25)
        ),
        yaxis=dict(
        ),
        
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        #choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )

    return fig

#=======plot_stress_test=============================================
def plot_stress_test():

    # Define scenarios, currencies, and portfolio
    scenarios = ['stress_covid', 'stress_inflation']
    currencies=['FCY', 'AZN']
    portfolios=['MORTGAGE BONDS', 'AFS PORTFOLIO', 'HTM PORTFOLIO']

    # Filter relevant data
    df = df_metrics_sum.loc[df_metrics_sum['metric'].isin(scenarios)]
    
    # create subplots titles
    subplot_titles = ['All Bonds'] + [
        f"{c} {p}"
        for c in currencies
        for p in portfolios
        if not (c == 'FCY' and p == 'MORTGAGE BONDS')
    ]
    subplot_titles=tuple(subplot_titles)

    # ticks
    step = 20
    tickvalues = dates_array[::step]
    
    # Create subplots
    fig=make_subplots(
        rows=2,cols=3,
        subplot_titles=(subplot_titles),
        horizontal_spacing=0.03, # space between subplots
        shared_xaxes=True,
        shared_yaxes=False   
    )
    
    # Define consistent color scheme
    color_map = {
        'stress_covid': 'red',
        'stress_inflation': 'orange',
    }

    # Plot aggregated data for all bonds in first subplot
    for scenario in scenarios:
        y_values = df.loc[df['metric']==scenario, dates_array].sum().values
        fig.add_trace(
            go.Scatter(
                name = dic_metrics.get(scenario, scenario),
                x=dates_array,
                y=y_values,
                mode='lines',
                line=dict(color=color_map.get(scenario)),
                showlegend=True
            ), row=1, col=1
        )
    
    # Plot data for each currency and portfolio combination
    subplot_index=0
    for currency in currencies:
        for portfolio in portfolios:
            # get position
            subplot_index += 1
            row = 1 if subplot_index <= 3 else 2
            col = subplot_index if row==1 else subplot_index-3
            # skip FCY Mortgage Bonds as they do not exist
            if (portfolio=='MORTGAGE BONDS' and currency=='FCY'):
                continue
            
            #plot data
            for scenario in scenarios:
                y_values =df.loc[
                    (df['metric']==scenario)&
                    (df['portfolio_type']==portfolio)&
                    (df['currency']==currency),
                dates_array].values.flatten()
                fig.add_trace(
                    go.Scatter(
                        name = dic_metrics.get(scenario, scenario),
                        x=dates_array,
                        y=y_values,
                        mode='lines',
                        line=dict(color=color_map.get(scenario)),
                        showlegend=False
                    ), row=row, col=col
                    )
    
    fig.update_layout(
        title=dict(
            text=f"Change in fair value under Custom Stress Scenarios",
            font=dict(size=25)
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        # choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )

    return fig

#=======plot_st_parameters_cs=============================================
def plot_st_parameters_cs():

    #++++++++++++code from Metrics_calculation.ipynb++++++++++++++++++++++
    db_path = 'arms_database.db'
    
    stress_scenario_dic = {
        'stress_covid': ['covid_mar_2020', 'az_covid_2020'],
        'stress_inflation': ['inflation_mar_2022', 'az_inflation_2022']
    }
    reverse_map = {
        scenario: group
        for group, scenarios in stress_scenario_dic.items()
        for scenario in scenarios
    }
    
    custom_scenarios = []
    for key in stress_scenario_dic:
        custom_scenarios.extend(stress_scenario_dic[key])
    
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()
        
    placeholders = ', '.join(['?'] * len(custom_scenarios))
    GET_SPREAD_PARAMETERS = f"""
        SELECT * 
        FROM stress_scenarios_for_credit_spread
        WHERE scenario_name in ({placeholders}) 
    """
    cursor.execute(GET_SPREAD_PARAMETERS, custom_scenarios)
    df_columns = [description[0] for description in cursor.description]
    df = pd.DataFrame(data=cursor.fetchall(), columns=df_columns)
    df['scenario_group'] = df['scenario_name'].map(reverse_map)

    connection.close()
    #---------------------------------------------------------------------
    
    scenarios = ['stress_inflation', 'stress_covid']
    rating_categories=['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC']

    # initiate plot
    fig=go.Figure()

    # Add values
    for scenario in scenarios:
        x_values = df['rating_category_sp'].unique()
        y_values = df.loc[
            (df['range_start_days']==0)&      #taking shortest time bucket
            (df['scenario_group']==scenario),
            'spread_change_bp'
        ]
        
        fig.add_trace(
            go.Bar(
                name=dic_metrics.get(scenario, scenario),
                x=x_values,
                y=y_values,
                text=y_values,
                textposition='outside',
                cliponaxis=False  #Prevent clipping
            )
        )
        
    fig.update_layout(
        title=dict(
            text=f"Change in credit spread",
            font=dict(size=25)
        ),
        barmode='group',
        xaxis=dict(
            type='category',
            categoryorder='array',
            categoryarray=rating_categories
        ),
        yaxis=dict(
            title='change, bp',
            side='left'
        ),
        legend=dict(
            orientation='v', 
            x=0,
            y=1,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(0,0,0,0)'  # transparent background
        ),

        # choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )

    return fig

#=======plot_st_parameters_yc=============================================
def plot_st_parameters_yc():

    #++++++++++++code from Metrics_calculation.ipynb++++++++++++++++++++++
    db_path = 'arms_database.db'
    
    stress_scenario_dic = {
        'stress_covid': ['covid_mar_2020', 'az_covid_2020'],
        'stress_inflation': ['inflation_mar_2022', 'az_inflation_2022']
    }
    reverse_map = {
        scenario: group
        for group, scenarios in stress_scenario_dic.items()
        for scenario in scenarios
    }
    
    custom_scenarios = []
    for key in stress_scenario_dic:
        custom_scenarios.extend(stress_scenario_dic[key])
    
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()
        
    placeholders = ', '.join(['?'] * len(custom_scenarios))
    GET_YIELD_PARAMETERS = f"""
        SELECT * 
        FROM stress_scenarios_for_risk_free_rate
        WHERE scenario_name in ({placeholders}) 
    """
    cursor.execute(GET_YIELD_PARAMETERS, custom_scenarios)
    df_columns = [description[0] for description in cursor.description]
    df = pd.DataFrame(data=cursor.fetchall(), columns=df_columns)
    df['scenario_group'] = df['scenario_name'].map(reverse_map)

    connection.close()
    #---------------------------------------------------------------------

    currencies = ['AZN', 'USD']
    scenarios = ['stress_inflation', 'stress_covid']
    tenor_categories=basel_buckets[:-3] # cut off '15Y' and further
    
    df['tenor'] = pd.Categorical(
        df['tenor'],
        categories=tenor_categories,
        ordered=True)
    df=df.sort_values(by='tenor')
    # initiate plot
    fig=go.Figure()

    # Add values
    for currency in currencies:
        for scenario in scenarios:
            # filtering data
            df_aux = df.loc[
                (df['currency']==currency)&      #taking shortest time bucket
                (df['scenario_group']==scenario)&
                (df['tenor'].isin(tenor_categories))
            ].copy()
            x_values = df_aux['tenor']
            y_values = df_aux['rate_change_pp']
            scenario_name = f"{currency} {dic_metrics.get(scenario, scenario)}"

            # plot data
            fig.add_trace(
                go.Scatter(
                    name=scenario_name,
                    x=x_values,
                    y=y_values,
                    mode='lines'
                )
            )
        
    fig.update_layout(
        title=dict(
            text=f"Change in yiled curve",
            font=dict(size=25)
        ),
        xaxis=dict(
            type='category',
        ),
        yaxis=dict(
            title='change, pp',
            side='left'
        ),
        legend=dict(
            orientation='h', 
            x=0.5, # center horizontally
            xanchor='center',
            y=-0.10,
            yanchor='top', 
            ),
        # choosing theme and make the plot transparent
        template = 'plotly_dark',
        plot_bgcolor='rgba(0, 0, 0, 0)',
        paper_bgcolor='rgba(0, 0, 0, 0)',
    )

    return fig
# ========================================================================================


#-------------------------------------------------------------------------------------------
end_time = time.time()
print(f"Execution time: {end_time - start_time:.4f} seconds")

C:\Users\AJ003230\AppData\Roaming\Python\Python312\site-packages\bond_pricing\simple_bonds.py:529: UserWarning: Module isda_daycounters is not installed.
Only 'simple' daycount (basically ACT/365) is available.
To use other daycounts, install isda_daycounters from
https://github.com/miradulo/isda_daycounters
  warn("Module isda_daycounters is not installed.\n"


Execution time: 3.5506 seconds


## Test Charts

In [4]:
# test charts 
#fig = plot_book_value_and_mod_duration('AZN')
#fig.show()

#fig = plot_cash_flow('HTM PORTFOLIO','FCY', fx_translation=True)
#fig.show()

#fig = plot_ratings_and_risk('FCY', 'all', 'absolute')
#fig.show()

#fig=plot_int_rate_risk ('HTM PORTFOLIO', 'FCY')
#fig.show()

#fig = plot_em_credit_spread (selected_dates, 'median', 'Bucket 2-3')
#fig.show()

#fig = plot_dm_credit_spread (selected_dates, 'median', 'Bucket 0-3')
#fig.show()

#fig = plot_par_yield_curve(selected_dates, 'AZN', [0, 1500])
#fig.show()

#fig = plot_pnl('AFS PORTFOLIO', 'USD')
#fig.show()

#fig = plot_reval_diff('HTM PORTFOLIO', 'FCY')
#fig.show()

#fig = plot_sector_exposure('FCY')
#fig.show()

#fig=plot_basel_shock_scenario ('AZN')
#fig.show()

#fig=plot_basel_shock_results()
#fig.show()

#fig=plot_stress_test()
#fig.show()

#fig=plot_st_parameters_cs()
#fig.show()

#fig=plot_st_parameters_yc()
#fig.show()

## Create tables

In [5]:
#===========df_book_value_and_mod_duration============================================

# define formats
format_func_mio = lambda x: f"{x / 1_000_000:,.0f}M" if pd.notnull(x) else ""
format_func_dec = lambda x: f"{x:,.2f}" if pd.notnull(x) else ""
format_func_int = lambda x: f"{x:,.0f}" if pd.notnull(x) else ""

def df_book_value_and_mod_duration(currency, metrics, format_func):

    """
    Generates a formatted DataFrame showing selected metrics for a given currency,
    including changes over time and formatted output.

    Parameters:
    - currency (str): Currency filter (e.g., 'FCY')
    - metrics (list): List of metric names (e.g., ['book_value', 'mod_duration'])
    - format_funcs (list): List of formatting functions corresponding to each metric

    Returns:
    - pd.DataFrame: Formatted and structured output
    """
    columns_to_take = ['metric', 'portfolio_type'] + selected_dates
    df = df_metrics_sum.loc[
        (df_metrics_sum['metric'].isin(metrics))&
        (df_metrics_sum['currency']==currency),
        columns_to_take
    ].copy()

    # Define portfolio order
    port_order = ['AFS PORTFOLIO', 'HTM PORTFOLIO', 'MORTGAGE BONDS']
    df['portfolio_type'] = pd.Categorical(
        df['portfolio_type'],
        categories=port_order,
        ordered=True)
    # calculate difference
    df['Δ 5 bd'] = df[columns_to_take[-1]] - df[columns_to_take[-2]]
    df['Δ 20 bd'] = df[columns_to_take[-1]] - df[columns_to_take[-3]]
    df['Δ PTD'] = df[columns_to_take[-1]] - df[columns_to_take[-4]]

    # Prepare final DataFrame
    column_names = list(df.columns)
    df_inter =pd.DataFrame(columns=column_names[1:]) #exclude 'metric'
    for metric,format_func in zip(metrics, format_func):
        df_aux = df.loc[df['metric']==metric,column_names[1:]].copy()
        df_aux = df_aux.astype(object) # Convert all values to object dtype before formatting
        df_aux.iloc[:, 1:] = df_aux.iloc[:, 1:].map(format_func)
        if dic_metrics[metric]:
            metric_name = dic_metrics[metric]
        else:
            metric_name = metric
        df_inter.loc[len(df_inter)] = [metric_name] + [""]*(len(df_inter.columns)-1)
        df_inter = pd.concat([df_inter, df_aux], ignore_index=True)
    
    df = df_inter
    df = df.rename(columns={'portfolio_type': 'PORTFOLIO'})
    
    return df

#===========df_cr_and_dv============================================================
def df_cr_and_dv(date, portfolio, currency, format_func):
    """
    portfolio as in report or 'all'
    currency as in report or 'FCY&AZN'/'all'
    """

    # get initial data
    dec_columns = ['MD']
    int_columns = ['cr01', 'dv01'] + [f"dv01_{tenor}" for tenor in basel_buckets]
    metrics_list = dec_columns + int_columns
    index_columns = ['portfolio_type', 'currency']
    aux_columns = ['fair_value', 'fx_rate'] 
    columns_to_take = index_columns + int_columns + aux_columns
    df = source_data.loc[source_data['RepDate']==date, columns_to_take]
    
    # fx translation
    df[int_columns]=df[int_columns].mul(df['fx_rate'], axis=0)
    df['fair_value']=df['fair_value'].mul(df['fx_rate'], axis=0)
    
    # adjust to portfolio aggregation request
    if portfolio!='all' and portfolio not in df['portfolio_type'].unique(): 
        print(f"unknown portfolio name: {portfolio}")
        return pd.DataFrame()
    if portfolio !='all':
       df = df.loc[df['portfolio_type']==portfolio] 
    
    # adjust to currency aggregation request
    if currency=='FCY&AZN':
        df.loc[df['currency']!='AZN', 'currency'] = 'FCY'    

    # filtering data
    if currency=='all' or currency=='FCY&AZN':
            pass
    elif currency!='FCY&AZN' and currency not in df['currency'].unique():
        print(f"unknown currency name: {currency}")
        return pd.DataFrame()
    else:
        df = df.loc[
            (df['portfolio_type']==portfolio)&
            (df['currency']==currency)
        ]
    # aggregate data
    columns_to_aggregate = int_columns + ['fair_value']
    df = df.groupby(index_columns, observed=True)[columns_to_aggregate].sum().reset_index()
    df['MD'] = -df['dv01']/df['fair_value']*10000
    #change order of columns
    df=df[index_columns+metrics_list]

    #convert values to string
    df[metrics_list] = df[metrics_list].astype(object) # Convert all values to object dtype before formatting
    df[dec_columns] = df[dec_columns].map(format_func[0])
    df[int_columns] = df[int_columns].map(format_func[1])
    
    return df

#===================================================================================
def df_new_invest(format_func):
    df=df_new_deals.copy()

    dec_columns=['fx_rate']
    int_columns = ['initial_notional', 'invested_amount', 'fair_value', 'invested_amount_azn']
    metrics_list = dec_columns + int_columns
    #convert values to string
    df[metrics_list] = df[metrics_list].astype(object) # Convert all values to object dtype before formatting
    df[dec_columns] = df[dec_columns].map(format_func[0])
    df[int_columns] = df[int_columns].map(format_func[1])

    return df

    

# test
#df = df_book_value_and_mod_duration('FCY', ['book_value', 'mod_duration'],[format_func_mio, format_func_dec])
#print(df)

#df = df_cr_and_dv(end_date, 'all', 'FCY&AZN',[format_func_dec, format_func_int])
#print(df)

#df = df_new_invest([format_func_dec, format_func_int])
#print(df)

## Test tables

## Create Dash Report

In [6]:
start_time = time.time()
#-------------------------------------------------------------------------------------------
#========PAGE SIZE & MARGINS===================================================================
# Version 1
# inital page size 1754 x 1240
# trying to fit into landscaoe A4, no margins, scale 65% 
page_margin = 30
page_width = 1754 - page_margin*2
page_height = 1240 - page_margin*2

page_size ={
    'width': f'{page_width}px', 
    'height':f'{page_height}px',
    'margin-top': f'{page_margin}px',
    'margin-bottom': f'{page_margin}px',
    'margin-left': f'{page_margin}px',
    'margin-right': f'{page_margin}px',
    #'margin':'auto', 
    #'border':'1px solid white'
}
#====== PAGE OVERVIEW ===============================================================
page_overview = dbc.Container([
        dbc.Row([
            dbc.Col(
                [html.H1(f'BOND PORTFOLIO OVERVIEW: {start_date} - {end_date}')],
                width=8, align="center"
            ),
            dbc.Col(
                [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
                width={"size": 3}, 
                    )
        ], style={'margin-top': 20,}, align="center", justify="between"
        ),
        html.Br(),
        dbc.Row([
            dbc.Col([dcc.Graph(figure = plot_book_value_and_mod_duration('AZN'))],
                   width=7),
            dbc.Col([dbc.Table.from_dataframe(
                df_book_value_and_mod_duration('AZN',['book_value', 'mod_duration'], [format_func_mio, format_func_dec]),
                responsive=False,
                style={
                    "width": "max-content",
                    "transform": "scale(0.90)",
                    "transformOrigin": "center left"
                    }
                )
                ],
                width=5, align="center"),            
            ],
        style={'height': '45%'}),
        html.Br(),
        dbc.Row([
            dbc.Col([dcc.Graph(figure = plot_book_value_and_mod_duration('FCY'))],
                width=7),
            dbc.Col([dbc.Table.from_dataframe(
                df_book_value_and_mod_duration('FCY',['book_value', 'mod_duration'], [format_func_mio, format_func_dec]),
                responsive=False,
                style={
                    "width": "max-content",
                    "transform": "scale(0.90)",
                    "transformOrigin": "center left"
                    }
                )],
                width=5, align="center"),            
            ],
        style={'height': '45%'})
    ],
    fluid=True, style=page_size)

#=======PAGE PORTFOLIO CASH FLOWS===============================================================================
page_cash_flows = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('BOND PORTFOLIO CASH FLOWS')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_cash_flow('AFS PORTFOLIO','AZN', fx_translation=True))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_cash_flow('AFS PORTFOLIO','FCY', fx_translation=True))],
               width=6),
        ], style={'height': '45%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_cash_flow('HTM PORTFOLIO','AZN', fx_translation=True))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_cash_flow('HTM PORTFOLIO','FCY', fx_translation=True))],
               width=6),
            ],style={'height': '45%'})
        ],
    fluid=True, style=page_size)

#=======PAGE PORTFOLIO CREDIT RISK===============================================================================
page_credit_risk = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('BOND PORTFOLIO CREDIT RISKS')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_ratings_and_risk('FCY', 'all', 'absolute'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_ratings_and_risk('FCY', 'all', 'relative'))],
               width=6),
        ], style={'height': '45%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_ratings_and_risk('AZN', 'all', 'absolute'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_ratings_and_risk('AZN', 'all', 'relative'))],
               width=6),
        ], style={'height': '45%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE PORTFOLIO INTEREST RATE RISK===============================================================================
page_int_rate_risk = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('BOND PORTFOLIO INTEREST RATE RISKS')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([
            dbc.Table.from_dataframe(
                df_cr_and_dv(end_date, 'all', 'FCY&AZN',[format_func_dec, format_func_int]),
                style={
                    "width": "max-content",
                    "transform": "scale(0.75)",
                    "transformOrigin": "center left"
                    }
            )  
        ], width='auto' )
    ], align="center", justify="center",
    style={'height': '22%'}),    
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_int_rate_risk ('AFS PORTFOLIO', 'FCY'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_int_rate_risk ('AFS PORTFOLIO', 'AZN'))],
               width=6),
        ], style={'height': '34%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_int_rate_risk ('HTM PORTFOLIO', 'FCY'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_int_rate_risk ('HTM PORTFOLIO', 'AZN'))],
               width=6),
        ], style={'height': '34%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE MARKET TRENDS===============================================================================
page_market_trends = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('MARKET: YIELD CURVES AND CREDIT SPREAD')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),    
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_par_yield_curve(selected_dates, 'USD', [0, 1500]))],
               width=4),
        dbc.Col([dcc.Graph(figure=plot_par_yield_curve(selected_dates, 'GBP', [0, 1500]))],
               width=4),
        dbc.Col([dcc.Graph(figure=plot_par_yield_curve(selected_dates, 'AZN', [0, 1500]))],
           width=4),
        ], className="g-0", style={'height': '45%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_dm_credit_spread (selected_dates, 'median', 'Bucket 0-3'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_em_credit_spread (selected_dates, 'median', 'Bucket 2-3'))],
               width=6),
        ], style={'height': '45%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE PORTFOLIO PnL and Return===============================================================================
page_pnl = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('BOND PORTFOLIO THEORETICAL PnL & RETURN')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_pnl('AFS PORTFOLIO', 'USD'))],
            width=4),
        dbc.Col([dcc.Graph(figure=plot_pnl('AFS PORTFOLIO', 'AZN'))],
            width=4),
        dbc.Col([dcc.Graph(figure=plot_pnl('AFS PORTFOLIO', 'GBP'))],
            width=4),
        ], style={'height': '45%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_pnl('AFS PORTFOLIO', 'EUR'))],
            width=4),
        dbc.Col([dcc.Graph(figure=plot_pnl('HTM PORTFOLIO', 'USD'))],
            width=4),
        dbc.Col([dcc.Graph(figure=plot_pnl('HTM PORTFOLIO', 'AZN'))],
            width=4),
        ], style={'height': '45%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE REVALUATION DIFFERENCE===============================================================================
page_reval_diff = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('BOND PORTFOLIO REVALUATION DIFFERENCE')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_reval_diff('HTM PORTFOLIO', 'FCY'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_reval_diff('HTM PORTFOLIO', 'AZN'))],
               width=6),
        ], style={'height': '45%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_reval_diff('AFS PORTFOLIO', 'FCY'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_reval_diff('AFS PORTFOLIO', 'AZN'))],
               width=6),
        ], style={'height': '45%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE SECTOR BREAKDOWN===============================================================================
page_sector_breakdown = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('BOND PORTFOLIO SECTOR BREAKDOWN')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_sector_exposure('FCY'))],
               width=12),
        ], style={'height': '60%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_sector_exposure('AZN'))],
               width=12),
        ], style={'height': '30%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE BASEL SHOCKS===============================================================================
page_basel_shocks = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H2("BONDS FAIR VALUE CHANGE UNDER BASEL'S RATE SHOCK SCENARIOS")],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_basel_shock_scenario ('USD'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_basel_shock_scenario ('AZN'))],
               width=6),
        ], style={'height': '35%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_basel_shock_results())],
               width=12),
        ], style={'height': '55%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE CUSTOM STRESS TEST===============================================================================
page_custom_shocks = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H2("BONDS FAIR VALUE CHANGE UNDER CUSTOM STRESS SCENARIOS")],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_st_parameters_cs())],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_st_parameters_yc())],
               width=6),
        ], style={'height': '35%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_stress_test())],
               width=12),
        ], style={'height': '55%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE MORTGAGE BONDS===============================================================================
page_mortgage_bonds = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1('MORTGAGE BONDS')],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_cash_flow('MORTGAGE BONDS','AZN', fx_translation=True))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_int_rate_risk ('MORTGAGE BONDS', 'AZN'))],
               width=6),
        ], style={'height': '45%'}),
    html.Br(),
    dbc.Row([
        dbc.Col([dcc.Graph(figure=plot_reval_diff('MORTGAGE BONDS', 'AZN'))],
               width=6),
        dbc.Col([dcc.Graph(figure=plot_pnl('MORTGAGE BONDS', 'AZN'))],
               width=6),
        ], style={'height': '45%'}),
        ],
    fluid=True, style=page_size)

#=======PAGE NEW DEALS===============================================================================
page_new_deals = dbc.Container([
    dbc.Row([
        dbc.Col(
            [html.H1(f"NEW DEALS FROM {prev_date1} TILL {end_date}")],
            width=8, align="center"
        ),
        dbc.Col(
            [html.Img(src="assets/logos/Pb_le_p1_rgb_pos.png", style={"height": "50px"})],
            width={"size": 3}, 
        )
    ], style={'margin-top': 20,}, align="center", justify="between"),
    html.Br(),
    dbc.Row([
        dbc.Col([
            dbc.Table.from_dataframe(
                df_new_invest([format_func_dec, format_func_int]),
                style={
                    "width": "max-content",
                    #"transform": "scale(0.75)",
                    "margin": "0 auto"
                    }
            )  
        ], width=11 )
    ], align="center", justify="center",
    style={'height': '90%'}),    

        ],
    fluid=True, style=page_size)

#===================================================================================================


#-------------------------------------------------------------------------------------------
end_time = time.time()

print(f"Block execution time: {end_time - start_time:.4f} seconds")


size of output data is (1, 24)
size of output data is (1, 24)
size of output data is (1, 24)
size of output data is (1, 24)
size of output data is (1, 21)
size of output data is (1, 21)
size of output data is (1, 21)
size of output data is (1, 21)
Block execution time: 3.6541 seconds


## Generate Report

In [7]:
start_time = time.time()
#-------------------------------------------------------------------------------------------

# create the Dash app
app = Dash(
    __name__,
    title="Bonds Report",
    external_stylesheets=[dbc.themes.SLATE])

# create app layout
app.layout = dbc.Container([
    page_overview,
    page_market_trends,
    page_new_deals,
    page_credit_risk,
    page_int_rate_risk,
    page_cash_flows,
    page_basel_shocks,
    page_custom_shocks,
    page_pnl,
    page_reval_diff,
    page_sector_breakdown,
    page_mortgage_bonds,
    ],
    fluid=True)

# run the app
if __name__=='__main__':
    #app.run(debug=True)
    #app.run(jupyter_mode="tab", port=8051) # automatically open the app in a new browser tab
    app.run(jupyter_mode="external", port=8052) # display a link to visit to view the app
#-------------------------------------------------------------------------------------------
end_time_total = time.time()
end_time = time.time()

print(f"Block execution time: {end_time - start_time:.4f} seconds")
print(f"Total execution time: {end_time_total - start_time_total:.4f} seconds")

Dash app running on http://127.0.0.1:8052/
Block execution time: 0.6411 seconds
Total execution time: 20.1789 seconds
